# RF Coach Vision: video per il dataset di TrackNet (sessione 2 di 3)

Notebook **solo per preparare il dataset** di addestramento di TrackNet: analizza **più video di fila** e salva su Drive i file che servono a `scegli_tratti_addestramento.py` (`_tracking.csv`, `_colpi.csv`, `_velocita.csv`, `_velocita_punti.json`). Per le analisi normali si usa sempre `rf_coach_colab.ipynb`, che resta com'è.

Le sessioni sono 3 (`rf_coach_dataset_1/2/3.ipynb`), ognuna con i suoi video: si possono aprire **insieme**, in tre schede, ognuna ha la sua macchina. Non aprire lo stesso notebook in due schede: userebbero la stessa macchina.

**Questa sessione:** `federer_TRAIN_VAL_2.mp4`, `roma_tennis_a_TRAIN_VAL_2.mp4`, `roma_tennis_b_TRAIN_VAL_2.mp4`

**Come si usa:** menu **Runtime → Cambia tipo di runtime → GPU T4**, poi **Runtime → Esegui tutto**. Le risposte servono solo all'inizio: l'autorizzazione di Drive (cella 2) e la calibrazione del campo dei video di questa sessione (cella 4b, un video dopo l'altro). Dopo va da sola: ogni video viene salvato su Drive appena finito, e alla fine la sezione 7 dice quali video hanno i colpi.

## 0. I video di questa sessione

In [ ]:
# I video di questa sessione (in Drive/rf_coach_vision/inputs). Per cambiarli basta modificare la lista.
LISTA = ["federer_TRAIN_VAL_2.mp4", "roma_tennis_a_TRAIN_VAL_2.mp4", "roma_tennis_b_TRAIN_VAL_2.mp4"]
TRACKNET_MODE = "weight"   # come nel notebook principale
RICALIBRA = False          # False: un video che ha gia' la sua calibrazione su Drive non la richiede
FOTOGRAMMA = {}            # fotogramma da calibrare per video, per esempio {"fognini_TRAIN_VAL_2.mp4": 120}; se manca, 0

## 1. Controllo GPU

In [ ]:
!nvidia-smi -L

## 2. Collega Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 3. Scarica il codice da GitHub

Il repository è **pubblico**: non serve nessun token.
Se un giorno tornasse **privato**, servirebbe un token: pannello a sinistra, icona della chiave (Secrets), un segreto chiamato `GITHUB_TOKEN` (token *fine-grained* con accesso al solo `rf_coach_vision` e permesso *Contents: Read-only*) con l'accesso al notebook attivo. La cella lo userà da sola.

In [ ]:
import os, shutil, glob, subprocess

REPO = "anrundo-2312/rf_coach_vision"
DRIVE_DIR = "/content/drive/MyDrive/rf_coach_vision"
WORK_DIR = "/content/rf_coach_vision"

# Il repo e' pubblico: il token serve solo se un giorno torna privato.
token = ""
try:
    from google.colab import userdata
    token = (userdata.get("GITHUB_TOKEN") or "").strip()
except Exception:
    pass
url = f"https://{REPO.split('/')[0]}:{token}@github.com/{REPO}.git" if token else f"https://github.com/{REPO}.git"

# Se la cella viene rieseguita, la cartella corrente e' quella che stiamo per
# cancellare (l'analisi fa %cd li'): git non partirebbe. Prima usciamo.
os.chdir("/content")
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
r = subprocess.run(["git", "clone", "--depth", "1", url, WORK_DIR], capture_output=True, text=True,
                   env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})
if r.returncode != 0:
    raise SystemExit("Download da GitHub non riuscito:\n" + (r.stderr.replace(token, "***") if token else r.stderr))
print("codice scaricato:", sorted(os.listdir(WORK_DIR)))

## 4. Porta dal Drive modelli, calibrazioni e i video della lista

In [ ]:
# Come la cella 4 del notebook principale, ma dei video copia solo quelli della LISTA (piu' veloce)
copie = [("models", ""), ("ckpts", "tracknet3/ckpts"), ("pred_result", "tracknet3/pred_result"),
         ("calibrazioni", "velocita/calibrazioni")]

for sub, dst in copie:
    src = os.path.join(DRIVE_DIR, sub)
    dst_dir = os.path.join(WORK_DIR, dst) if dst else WORK_DIR
    os.makedirs(dst_dir, exist_ok=True)
    if os.path.isdir(src):
        for f in glob.glob(os.path.join(src, "*")):
            if os.path.isfile(f):
                shutil.copy(f, dst_dir)

os.makedirs(os.path.join(WORK_DIR, "inputs"), exist_ok=True)
mancano_video = [v for v in LISTA if not os.path.exists(os.path.join(DRIVE_DIR, "inputs", v))]
assert not mancano_video, f"Non ci sono in Drive/rf_coach_vision/inputs: {mancano_video}"
for v in LISTA:
    shutil.copy(os.path.join(DRIVE_DIR, "inputs", v), os.path.join(WORK_DIR, "inputs"))

mancanti = [f for f in ["tennis_yolo11.pt", "yolo11n-pose.pt", "tracknet3/ckpts/TrackNet_best.pt"]
            if not os.path.exists(os.path.join(WORK_DIR, f))]
assert not mancanti, f"Mancano su Drive: {mancanti} (vedi la tabella in cima al notebook principale)"

print("video di questa sessione:", sorted(os.listdir(os.path.join(WORK_DIR, "inputs"))))

## 4b. Calibrazione del campo, un video dopo l'altro

Per ogni video della lista: se ha gia' la sua calibrazione (`calibrazioni/<video>.json` su Drive) la mostra e passa al successivo; se no compare il fotogramma da calibrare. Questi video non sono ripresi come il telefono di SwingVision, quindi la calibrazione **standard** non va bene: si calibra sempre.

Stessi comandi del notebook principale: **clic** sul fotogramma = segna il punto; **clic nella lente** = corregge l'ultimo punto; **frecce** = spostano l'ultimo punto di 1 pixel; **S** = punto non visibile, salta; **U** = annulla; **F** = fine (almeno 6 punti, sparsi: vicino, rete, lontano).

Se in un video il giocatore copre le righe: scrivi un altro fotogramma per quel video in `FOTOGRAMMA` (cella 0) e riesegui questa cella. Le calibrazioni vanno anche su Drive e valgono per le prossime sessioni.

In [ ]:
import sys, cv2
from IPython.display import Image, display
%cd {WORK_DIR}
sys.path.insert(0, os.path.join(WORK_DIR, "velocita"))
from calibra_campo import carica_per_video, disegna_controllo, leggi_fotogramma
import calibra_colab

def mostra_campo(video, fotogramma):
    cal, fonte = carica_per_video(f"inputs/{video}", None)
    frame, _, _ = leggi_fotogramma(f"inputs/{video}", fotogramma)
    testo = ("calibrazione STANDARD" if fonte == "standard"
             else f"calibrazione del video (errore medio {cal['errore_rms_px']:.1f} px)")
    cv2.imwrite("/content/campo_controllo.jpg", disegna_controllo(frame, cal, {}, f"{video}: {testo}"))
    display(Image("/content/campo_controllo.jpg", width=900))
    return fonte

for VIDEO in LISTA:
    f0 = FOTOGRAMMA.get(VIDEO, 0)
    _, fonte = carica_per_video(f"inputs/{VIDEO}", None)
    if fonte != "standard" and not RICALIBRA:
        print(f"\n{VIDEO}: ha gia' la sua calibrazione")
        mostra_campo(VIDEO, f0)
        continue
    print(f"\n========== {VIDEO}: calibra il campo (fotogramma {f0})")
    calibra_colab.calibra(f"inputs/{VIDEO}", f0, cartella_drive=DRIVE_DIR)
    mostra_campo(VIDEO, f0)

print("\nCalibrazioni fatte. Da qui in poi non serve toccare niente.")

## 5. Installa le librerie (PyTorch c'è già su Colab)

In [ ]:
!pip install -q ultralytics parse

## 6. Analisi di tutti i video della lista, uno dopo l'altro

Per ogni video: analisi (`analyze.py`), classificazione dei colpi, velocita' d'uscita; poi salva subito su Drive (`outputs/dati` e `pred_result`) i file di quel video, cosi' se la sessione cade quelli finiti restano. Non fa i video con le scritte sopra (per il dataset non servono): per guardarli si usa il notebook principale.

In [ ]:
import re, time
%cd {WORK_DIR}
p = os.path.join(WORK_DIR, "analyze.py")
s = open(p).read()
s = re.sub(r"^TRACKNET_MODE = .*$", f'TRACKNET_MODE = "{TRACKNET_MODE}"', s, flags=re.M)
s = re.sub(r"^TRACKNET_FORCE_RECOMPUTE = .*$", "TRACKNET_FORCE_RECOMPUTE = False", s, flags=re.M)
s = re.sub(r"^TRACKNET_PESI = .*$", "TRACKNET_PESI = ''", s, flags=re.M)
open(p, "w").write(s)

ANNOTAZIONI = os.path.join(DRIVE_DIR, "datasets/tennis_actions/Tennis Player Actions Dataset for Human Pose Estimation/annotations")
if os.path.isdir(ANNOTAZIONI):
    !python classificazione/addestra_colpi.py --annotazioni "{ANNOTAZIONI}"
else:
    print("Annotazioni non trovate su Drive: uso il modello salvato nel repo.")

t_tutto = time.time()
for VIDEO in LISTA:
    NOME = VIDEO.replace(".mp4", "")
    TRACKING = f"outputs/dati/{NOME}_tracking.csv"
    t0 = time.time()
    print(f"\n========== {VIDEO}")
    !python analyze.py inputs/{VIDEO}
    !python classificazione/classifica_tracking.py --tracking {TRACKING}
    !python classificazione/rileva_impatti.py --tracking {TRACKING}
    if os.path.exists(f"outputs/dati/{NOME}_impatti.csv"):
        !python classificazione/metriche_colpi.py --tracking {TRACKING}
    !python velocita/velocita_uscita.py --video inputs/{VIDEO}
    !python velocita/direzione_nascosta.py --video inputs/{VIDEO}
    !python velocita/velocita_rimbalzo.py --video inputs/{VIDEO}
    for sub, src in [("outputs/dati", "outputs/dati"), ("pred_result", "tracknet3/pred_result")]:
        dst = os.path.join(DRIVE_DIR, sub)
        os.makedirs(dst, exist_ok=True)
        for f in glob.glob(os.path.join(WORK_DIR, src, NOME + "*")):
            if os.path.isfile(f):
                shutil.copy(f, dst)
    print(f"{VIDEO}: {(time.time() - t0) / 60:.1f} minuti, salvato su Drive")

print(f"\nTempo totale: {(time.time() - t_tutto) / 60:.1f} minuti")

## 7. Riepilogo

Video con i colpi (`_velocita.csv`) e video senza. Un video senza colpi non si perde: `scegli_tratti_addestramento.py` ne prende solo tratti a caso.

In [ ]:
import pandas as pd
for v in LISTA:
    p = os.path.join(DRIVE_DIR, "outputs", "dati", v[:-4] + "_velocita.csv")
    n = len(pd.read_csv(p)) if os.path.exists(p) else 0
    print(f"{v}: {n} colpi" + ("" if n else "  <-- nessun colpo trovato"))